# TP2: Régression logistique à la main

**IFT6390 - Fondements de l'apprentissage machine**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pierrelux/mlbook/blob/main/exercises/tp2_logistic_regression.ipynb)

Ce notebook accompagne le [Chapitre 3: Classification linéaire](https://pierrelux.github.io/mlbook/ch3_classification).

## Objectifs

À la fin de ce TP, vous serez en mesure de:
- Implémenter la fonction sigmoïde et comprendre son rôle
- Calculer l'entropie croisée binaire
- Dériver et implémenter le gradient de la perte logistique
- Entraîner un classifieur par descente de gradient
- Visualiser la frontière de décision apprise

Ce TP implémente **tout à la main**, sans utiliser scikit-learn pour l'entraînement. L'objectif est de comprendre chaque étape du processus.

---

## Partie 0: Configuration

Exécutez cette cellule pour importer les bibliothèques nécessaires.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Pour de jolis graphiques
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

print("Configuration terminée!")

---
## Partie 1: Les données

Nous commençons par générer un jeu de données de classification binaire en 2D. Deux groupes de points, un pour chaque classe, avec un léger chevauchement.

**Question préliminaire**: Pouvez-vous séparer ces deux classes avec une droite?

In [ ]:
# Génération des données
np.random.seed(42)
n_per_class = 75

# Classe 0: centrée en (-1.5, -1.5)
X0 = np.random.randn(n_per_class, 2) * 0.9 + np.array([-1.5, -1.5])
# Classe 1: centrée en (1.5, 1.5)
X1 = np.random.randn(n_per_class, 2) * 0.9 + np.array([1.5, 1.5])

X = np.vstack([X0, X1])
y = np.array([0] * n_per_class + [1] * n_per_class)

print(f"Nombre d'exemples: {len(X)}")
print(f"Dimension des entrées: {X.shape[1]}")
print(f"Classes: {np.unique(y)}")

In [ ]:
# Visualisation des données
plt.figure(figsize=(7, 6))
plt.scatter(X0[:, 0], X0[:, 1], c='C1', s=50, label='Classe 0', alpha=0.7, edgecolors='white')
plt.scatter(X1[:, 0], X1[:, 1], c='C0', s=50, label='Classe 1', alpha=0.7, edgecolors='white')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('Données de classification binaire')
plt.legend()
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.show()

### Ajout du biais

Pour la régression logistique, nous modélisons:

$$p(y=1|\mathbf{x}) = \sigma(\theta_0 + \theta_1 x_1 + \theta_2 x_2)$$

Pour simplifier la notation, nous ajoutons une colonne de 1 aux données. Ainsi, le produit $\boldsymbol{\theta}^\top \mathbf{x}$ inclut automatiquement le biais.

In [ ]:
# Ajouter une colonne de 1 pour le biais
X_bias = np.column_stack([np.ones(len(X)), X])

print(f"Forme de X avec biais: {X_bias.shape}")
print(f"Premières lignes:")
print(X_bias[:3])

---
## Partie 2: La fonction sigmoïde

La **fonction sigmoïde** transforme n'importe quel nombre réel en une valeur entre 0 et 1:

$$\sigma(a) = \frac{1}{1 + e^{-a}}$$

C'est la clé de la régression logistique: elle permet d'interpréter la sortie comme une probabilité.

### Exercice 1: Implémenter la sigmoïde ★

Complétez la fonction ci-dessous.

In [ ]:
def sigmoid(a):
    """
    Calcule la fonction sigmoïde.
    
    Args:
        a: scalaire ou array numpy
    
    Returns:
        sigma(a) = 1 / (1 + exp(-a))
    """
    # ============================================
    # TODO: Implémentez la sigmoïde
    # Indice: utilisez np.exp()
    # ============================================
    
    result = None  # <- Remplacez par votre code
    
    return result

In [ ]:
# Test de votre fonction
if sigmoid(0) is not None:
    print(f"sigma(0) = {sigmoid(0):.4f}  (attendu: 0.5)")
    print(f"sigma(2) = {sigmoid(2):.4f}  (attendu: 0.8808)")
    print(f"sigma(-2) = {sigmoid(-2):.4f}  (attendu: 0.1192)")
    
    # Vérification
    if np.isclose(sigmoid(0), 0.5) and np.isclose(sigmoid(2), 0.8808, atol=1e-3):
        print("\nCorrect!")
    else:
        print("\nVérifiez votre implémentation.")
else:
    print("Complétez la fonction sigmoid!")

<details>
<summary><b>Solution</b> (cliquez pour afficher)</summary>

```python
def sigmoid(a):
    return 1 / (1 + np.exp(-a))
```
</details>

### Visualiser la sigmoïde

Observons comment la sigmoïde transforme le score $a = \boldsymbol{\theta}^\top \mathbf{x}$ en probabilité.

In [ ]:
# Visualisation de la sigmoïde
a_values = np.linspace(-6, 6, 200)

if sigmoid(0) is not None:
    plt.figure(figsize=(8, 5))
    plt.plot(a_values, sigmoid(a_values), 'C0-', linewidth=2)
    plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
    plt.axvline(0, color='gray', linestyle='--', alpha=0.5)
    
    plt.xlabel('$a = \\boldsymbol{\\theta}^\\top \\mathbf{x}$')
    plt.ylabel('$\\sigma(a) = P(y=1|\\mathbf{x})$')
    plt.title('La fonction sigmoïde')
    
    # Annotations
    plt.annotate('$a < 0$: classe 0 plus probable', xy=(-4.5, 0.15), fontsize=10, color='C1')
    plt.annotate('$a > 0$: classe 1 plus probable', xy=(1.5, 0.85), fontsize=10, color='C0')
    
    plt.xlim(-6, 6)
    plt.ylim(-0.05, 1.05)
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("Complétez d'abord la fonction sigmoid!")

---
## Partie 3: L'entropie croisée

La **perte** mesure à quel point nos prédictions sont mauvaises. Pour la régression logistique, nous utilisons l'**entropie croisée binaire**:

$$\mathcal{L}(\boldsymbol{\theta}) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\mu_i) + (1-y_i) \log(1-\mu_i) \right]$$

où $\mu_i = \sigma(\boldsymbol{\theta}^\top \mathbf{x}_i)$ est la probabilité prédite pour la classe 1.

**Intuition**: Si $y_i = 1$, on veut $\mu_i$ proche de 1, donc $\log(\mu_i)$ proche de 0. Si $y_i = 0$, on veut $\mu_i$ proche de 0, donc $\log(1-\mu_i)$ proche de 0.

### Exercice 2: Implémenter l'entropie croisée ★

Complétez la fonction ci-dessous.

**Conseil**: Utilisez `np.clip()` pour éviter les problèmes numériques avec $\log(0)$.

In [ ]:
def cross_entropy_loss(y_true, y_pred):
    """
    Calcule l'entropie croisée binaire moyenne.
    
    Args:
        y_true: étiquettes vraies (0 ou 1), array de taille (N,)
        y_pred: probabilités prédites (entre 0 et 1), array de taille (N,)
    
    Returns:
        Entropie croisée moyenne (scalaire)
    """
    # Éviter log(0) en "clipant" les probabilités
    eps = 1e-10
    y_pred = np.clip(y_pred, eps, 1 - eps)
    
    # ============================================
    # TODO: Calculez l'entropie croisée
    # Formule: -mean(y * log(pred) + (1-y) * log(1-pred))
    # ============================================
    
    loss = None  # <- Remplacez par votre code
    
    return loss

In [ ]:
# Test de votre fonction
y_test = np.array([1, 0, 1, 0])
pred_parfait = np.array([0.99, 0.01, 0.99, 0.01])
pred_mauvais = np.array([0.01, 0.99, 0.01, 0.99])
pred_neutre = np.array([0.5, 0.5, 0.5, 0.5])

if cross_entropy_loss(y_test, pred_parfait) is not None:
    print(f"Perte (prédictions parfaites): {cross_entropy_loss(y_test, pred_parfait):.4f}  (proche de 0)")
    print(f"Perte (prédictions neutres): {cross_entropy_loss(y_test, pred_neutre):.4f}  (environ 0.69)")
    print(f"Perte (prédictions inversées): {cross_entropy_loss(y_test, pred_mauvais):.4f}  (très élevée)")
    
    if cross_entropy_loss(y_test, pred_parfait) < 0.1:
        print("\nCorrect!")
else:
    print("Complétez la fonction cross_entropy_loss!")

<details>
<summary><b>Solution</b> (cliquez pour afficher)</summary>

```python
def cross_entropy_loss(y_true, y_pred):
    eps = 1e-10
    y_pred = np.clip(y_pred, eps, 1 - eps)
    
    loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return loss
```
</details>

### Visualiser la perte en fonction de la prédiction

Observons comment la perte pénalise les mauvaises prédictions.

In [ ]:
if cross_entropy_loss(y_test, pred_parfait) is not None:
    mu_values = np.linspace(0.01, 0.99, 100)
    
    # Perte quand y = 1
    loss_y1 = -np.log(mu_values)
    # Perte quand y = 0
    loss_y0 = -np.log(1 - mu_values)
    
    plt.figure(figsize=(8, 5))
    plt.plot(mu_values, loss_y1, 'C0-', linewidth=2, label='$y = 1$')
    plt.plot(mu_values, loss_y0, 'C1-', linewidth=2, label='$y = 0$')
    
    plt.xlabel('Probabilité prédite $\\mu$')
    plt.ylabel('Perte $\\ell$')
    plt.title('Entropie croisée: mauvaises prédictions = perte élevée')
    plt.legend()
    plt.ylim(0, 5)
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print("Pour y=1: si on prédit μ proche de 0, la perte explose.")
    print("Pour y=0: si on prédit μ proche de 1, la perte explose.")
else:
    print("Complétez d'abord les fonctions précédentes!")

---
## Partie 4: Le gradient

Pour minimiser la perte par descente de gradient, nous avons besoin du **gradient**. La dérivation (voir le chapitre 3) donne une formule simple:

$$\nabla_{\boldsymbol{\theta}} \mathcal{L} = \frac{1}{N} \sum_{i=1}^{N} (\mu_i - y_i) \mathbf{x}_i = \frac{1}{N} \mathbf{X}^\top (\boldsymbol{\mu} - \mathbf{y})$$

Le gradient est une moyenne pondérée des entrées, où le poids est **l'erreur de prédiction** $\mu_i - y_i$.

### Exercice 3: Implémenter le gradient ★★

Complétez la fonction ci-dessous.

In [ ]:
def compute_gradient(X, y, theta):
    """
    Calcule le gradient de l'entropie croisée.
    
    Args:
        X: matrice de données avec biais (N, d)
        y: étiquettes (N,)
        theta: paramètres actuels (d,)
    
    Returns:
        Gradient de la perte (d,)
    """
    N = len(y)
    
    # ============================================
    # TODO: 
    # 1. Calculer les probabilités mu = sigmoid(X @ theta)
    # 2. Calculer le gradient = X.T @ (mu - y) / N
    # ============================================
    
    mu = None  # <- Prédictions
    gradient = None  # <- Gradient
    
    return gradient

In [ ]:
# Test de votre fonction
theta_test = np.zeros(3)  # Initialisation à zéro

if sigmoid(0) is not None:
    grad = compute_gradient(X_bias, y, theta_test)
    
    if grad is not None:
        print(f"Gradient avec theta = [0, 0, 0]: {grad}")
        print(f"\nNorme du gradient: {np.linalg.norm(grad):.4f}")
        
        # Vérification: le gradient devrait pousser theta vers la séparation des classes
        # Classe 1 est en haut à droite, donc grad[1] et grad[2] devraient être négatifs
        # (car mu = 0.5 pour tous, et y=1 pour classe haute => mu - y = -0.5 pour classe 1)
        if grad[1] < 0 and grad[2] < 0:
            print("Le gradient pointe dans la bonne direction!")
    else:
        print("Complétez la fonction compute_gradient!")
else:
    print("Complétez d'abord la fonction sigmoid!")

<details>
<summary><b>Solution</b> (cliquez pour afficher)</summary>

```python
def compute_gradient(X, y, theta):
    N = len(y)
    mu = sigmoid(X @ theta)
    gradient = X.T @ (mu - y) / N
    return gradient
```
</details>

---
## Partie 5: Descente de gradient

Nous avons tous les ingrédients! La **descente de gradient** met à jour les paramètres de façon itérative:

$$\boldsymbol{\theta}_{t+1} = \boldsymbol{\theta}_t - \eta \cdot \nabla_{\boldsymbol{\theta}} \mathcal{L}(\boldsymbol{\theta}_t)$$

où $\eta > 0$ est le **taux d'apprentissage**.

### Exercice 4: Implémenter la descente de gradient ★★

Complétez la boucle d'entraînement.

In [ ]:
def train_logistic_regression(X, y, lr=0.1, n_iterations=100):
    """
    Entraîne un classifieur logistique par descente de gradient.
    
    Args:
        X: matrice de données avec biais (N, d)
        y: étiquettes (N,)
        lr: taux d'apprentissage
        n_iterations: nombre d'itérations
    
    Returns:
        theta: paramètres appris (d,)
        losses: historique de la perte
    """
    d = X.shape[1]
    theta = np.zeros(d)  # Initialisation
    losses = []
    
    for t in range(n_iterations):
        # ============================================
        # TODO:
        # 1. Calculer les probabilités prédites
        # 2. Calculer la perte (pour l'historique)
        # 3. Calculer le gradient
        # 4. Mettre à jour theta
        # ============================================
        
        # Probabilités prédites
        mu = None  # <- Complétez
        
        # Perte actuelle
        loss = None  # <- Complétez
        losses.append(loss)
        
        # Gradient
        grad = None  # <- Complétez
        
        # Mise à jour: theta = theta - lr * grad
        # <- Complétez
        pass
    
    return theta, losses

In [ ]:
# Entraînement!
if sigmoid(0) is not None and cross_entropy_loss(y_test, pred_parfait) is not None:
    theta_learned, loss_history = train_logistic_regression(
        X_bias, y, lr=0.5, n_iterations=100
    )
    
    if loss_history[0] is not None:
        print(f"Paramètres appris: biais={theta_learned[0]:.3f}, θ₁={theta_learned[1]:.3f}, θ₂={theta_learned[2]:.3f}")
        print(f"\nPerte initiale: {loss_history[0]:.4f}")
        print(f"Perte finale: {loss_history[-1]:.4f}")
    else:
        print("Complétez la fonction train_logistic_regression!")
else:
    print("Complétez d'abord les fonctions précédentes!")

<details>
<summary><b>Solution</b> (cliquez pour afficher)</summary>

```python
def train_logistic_regression(X, y, lr=0.1, n_iterations=100):
    d = X.shape[1]
    theta = np.zeros(d)
    losses = []
    
    for t in range(n_iterations):
        # Probabilités prédites
        mu = sigmoid(X @ theta)
        
        # Perte actuelle
        loss = cross_entropy_loss(y, mu)
        losses.append(loss)
        
        # Gradient
        grad = compute_gradient(X, y, theta)
        
        # Mise à jour
        theta = theta - lr * grad
    
    return theta, losses
```
</details>

### Visualiser la convergence

Traçons l'évolution de la perte au cours de l'entraînement.

In [ ]:
if 'loss_history' in dir() and loss_history[0] is not None:
    plt.figure(figsize=(8, 5))
    plt.plot(loss_history, 'C0-', linewidth=2)
    plt.xlabel('Itération')
    plt.ylabel('Entropie croisée')
    plt.title('Convergence de la descente de gradient')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("Complétez d'abord la fonction d'entraînement!")

---
## Partie 6: Prédiction et frontière de décision

Pour prédire, nous classifions dans la classe 1 si $p(y=1|\mathbf{x}) > 0.5$, c'est-à-dire si $\boldsymbol{\theta}^\top \mathbf{x} > 0$.

### Exercice 5: Implémenter la prédiction ★

In [ ]:
def predict_proba(X, theta):
    """Retourne les probabilités de la classe 1."""
    # ============================================
    # TODO: Calculez sigmoid(X @ theta)
    # ============================================
    return None  # <- Remplacez


def predict(X, theta, threshold=0.5):
    """Retourne les prédictions binaires (0 ou 1)."""
    # ============================================
    # TODO: Retournez 1 si proba >= threshold, 0 sinon
    # Indice: (predict_proba(X, theta) >= threshold).astype(int)
    # ============================================
    return None  # <- Remplacez

In [ ]:
# Évaluation
if 'theta_learned' in dir() and predict(X_bias, theta_learned) is not None:
    y_pred = predict(X_bias, theta_learned)
    accuracy = np.mean(y_pred == y)
    print(f"Précision sur les données d'entraînement: {accuracy:.1%}")
    
    # Matrice de confusion simple
    print(f"\nCorrectement classifiés: {np.sum(y_pred == y)} / {len(y)}")
else:
    print("Complétez les fonctions de prédiction!")

<details>
<summary><b>Solution</b> (cliquez pour afficher)</summary>

```python
def predict_proba(X, theta):
    return sigmoid(X @ theta)

def predict(X, theta, threshold=0.5):
    return (predict_proba(X, theta) >= threshold).astype(int)
```
</details>

### Visualiser la frontière de décision

La **frontière de décision** est la droite où $\boldsymbol{\theta}^\top \mathbf{x} = 0$. Les points d'un côté sont classés 0, de l'autre côté classés 1.

In [ ]:
if 'theta_learned' in dir() and loss_history[0] is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # --- Gauche: données et frontière ---
    ax = axes[0]
    ax.scatter(X0[:, 0], X0[:, 1], c='C1', s=50, label='Classe 0', alpha=0.7, edgecolors='white')
    ax.scatter(X1[:, 0], X1[:, 1], c='C0', s=50, label='Classe 1', alpha=0.7, edgecolors='white')
    
    # Frontière: theta[0] + theta[1]*x1 + theta[2]*x2 = 0
    # => x2 = -(theta[0] + theta[1]*x1) / theta[2]
    x1_range = np.linspace(-4, 5, 100)
    if np.abs(theta_learned[2]) > 1e-6:  # Éviter division par zéro
        x2_boundary = -(theta_learned[0] + theta_learned[1] * x1_range) / theta_learned[2]
        ax.plot(x1_range, x2_boundary, 'k-', linewidth=2, label='Frontière apprise')
    
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title('Classification par régression logistique')
    ax.set_xlim(-4, 5)
    ax.set_ylim(-4, 5)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    # --- Droite: carte de probabilité ---
    ax = axes[1]
    
    # Créer une grille
    xx, yy = np.meshgrid(np.linspace(-4, 5, 100), np.linspace(-4, 5, 100))
    X_grid = np.column_stack([np.ones(xx.ravel().shape), xx.ravel(), yy.ravel()])
    proba_grid = predict_proba(X_grid, theta_learned).reshape(xx.shape)
    
    # Carte de chaleur
    im = ax.contourf(xx, yy, proba_grid, levels=20, cmap='RdBu_r', alpha=0.8)
    ax.contour(xx, yy, proba_grid, levels=[0.5], colors='black', linewidths=2)
    
    ax.scatter(X0[:, 0], X0[:, 1], c='C1', s=30, alpha=0.7, edgecolors='white')
    ax.scatter(X1[:, 0], X1[:, 1], c='C0', s=30, alpha=0.7, edgecolors='white')
    
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title('Probabilité $P(y=1|\\mathbf{x})$')
    ax.set_xlim(-4, 5)
    ax.set_ylim(-4, 5)
    ax.set_aspect('equal')
    
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Probabilité')
    
    plt.tight_layout()
    plt.show()
else:
    print("Complétez d'abord les fonctions précédentes!")

---
## Partie 7: Expérimentations ★★

Explorons l'effet des hyperparamètres.

### 7.1 Effet du taux d'apprentissage

Le taux d'apprentissage $\eta$ contrôle la vitesse de convergence.

In [ ]:
if sigmoid(0) is not None and cross_entropy_loss(y_test, pred_parfait) is not None:
    learning_rates = [0.01, 0.1, 0.5, 1.0, 2.0]
    
    plt.figure(figsize=(10, 5))
    
    for lr in learning_rates:
        _, losses = train_logistic_regression(X_bias, y, lr=lr, n_iterations=100)
        if losses[0] is not None:
            plt.plot(losses, label=f'η = {lr}', linewidth=2)
    
    plt.xlabel('Itération')
    plt.ylabel('Entropie croisée')
    plt.title('Effet du taux d\'apprentissage')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 1)
    plt.show()
    
    print("Observations:")
    print("- η trop petit: convergence lente")
    print("- η trop grand: oscillations ou divergence")
else:
    print("Complétez d'abord les fonctions!")

### 7.2 Données non séparables

Que se passe-t-il si les classes se chevauchent davantage?

In [ ]:
if sigmoid(0) is not None:
    # Générer des données plus chevauchantes
    np.random.seed(123)
    n_overlap = 100
    
    X0_overlap = np.random.randn(n_overlap, 2) * 1.2 + np.array([-0.5, -0.5])
    X1_overlap = np.random.randn(n_overlap, 2) * 1.2 + np.array([0.5, 0.5])
    X_overlap = np.vstack([X0_overlap, X1_overlap])
    y_overlap = np.array([0] * n_overlap + [1] * n_overlap)
    X_overlap_bias = np.column_stack([np.ones(2 * n_overlap), X_overlap])
    
    # Entraîner
    theta_overlap, losses_overlap = train_logistic_regression(
        X_overlap_bias, y_overlap, lr=0.5, n_iterations=100
    )
    
    if losses_overlap[0] is not None:
        # Visualiser
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        ax = axes[0]
        ax.scatter(X0_overlap[:, 0], X0_overlap[:, 1], c='C1', s=50, label='Classe 0', alpha=0.6)
        ax.scatter(X1_overlap[:, 0], X1_overlap[:, 1], c='C0', s=50, label='Classe 1', alpha=0.6)
        
        x1_range = np.linspace(-4, 4, 100)
        if np.abs(theta_overlap[2]) > 1e-6:
            x2_boundary = -(theta_overlap[0] + theta_overlap[1] * x1_range) / theta_overlap[2]
            ax.plot(x1_range, x2_boundary, 'k-', linewidth=2, label='Frontière')
        
        ax.set_xlabel('$x_1$')
        ax.set_ylabel('$x_2$')
        ax.set_title('Données avec chevauchement')
        ax.legend()
        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        
        ax = axes[1]
        ax.plot(losses_overlap, 'C0-', linewidth=2)
        ax.set_xlabel('Itération')
        ax.set_ylabel('Entropie croisée')
        ax.set_title('Convergence')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Précision
        y_pred_overlap = predict(X_overlap_bias, theta_overlap)
        accuracy = np.mean(y_pred_overlap == y_overlap)
        print(f"\nPrécision avec données chevauchantes: {accuracy:.1%}")
        print("Le modèle fait de son mieux, mais ne peut pas séparer parfaitement.")
else:
    print("Complétez d'abord les fonctions!")

**Questions de réflexion:**
1. Pourquoi la perte ne descend-elle pas à zéro avec les données chevauchantes?
2. La frontière apprise est-elle raisonnable malgré les erreurs?
3. Comment pourrait-on améliorer la séparation? (Indice: caractéristiques non linéaires)

---
## Partie 8: Comparaison avec scikit-learn ★

Vérifions que notre implémentation donne des résultats similaires à scikit-learn.

In [ ]:
from sklearn.linear_model import LogisticRegression as SklearnLogReg

if 'theta_learned' in dir() and loss_history[0] is not None:
    # Entraîner avec scikit-learn
    model_sklearn = SklearnLogReg(penalty=None, max_iter=1000)
    model_sklearn.fit(X, y)  # X sans biais, sklearn l'ajoute
    
    # Comparer les coefficients
    print("Comparaison des paramètres:")
    print(f"\nNotre implémentation:")
    print(f"  biais = {theta_learned[0]:.4f}")
    print(f"  θ₁ = {theta_learned[1]:.4f}")
    print(f"  θ₂ = {theta_learned[2]:.4f}")
    
    print(f"\nScikit-learn:")
    print(f"  biais = {model_sklearn.intercept_[0]:.4f}")
    print(f"  θ₁ = {model_sklearn.coef_[0, 0]:.4f}")
    print(f"  θ₂ = {model_sklearn.coef_[0, 1]:.4f}")
    
    # Comparer les prédictions
    y_pred_ours = predict(X_bias, theta_learned)
    y_pred_sklearn = model_sklearn.predict(X)
    agreement = np.mean(y_pred_ours == y_pred_sklearn)
    
    print(f"\nAccord des prédictions: {agreement:.1%}")
else:
    print("Complétez d'abord les fonctions précédentes!")

---
## Récapitulatif

Dans ce TP, vous avez implémenté la régression logistique de A à Z:

1. **Sigmoïde**: $\sigma(a) = \frac{1}{1 + e^{-a}}$ transforme un score en probabilité

2. **Entropie croisée**: La perte qui pénalise les mauvaises prédictions
   $$\mathcal{L} = -\frac{1}{N} \sum_i \left[ y_i \log(\mu_i) + (1-y_i) \log(1-\mu_i) \right]$$

3. **Gradient**: Direction pour réduire la perte
   $$\nabla_{\boldsymbol{\theta}} \mathcal{L} = \frac{1}{N} \mathbf{X}^\top (\boldsymbol{\mu} - \mathbf{y})$$

4. **Descente de gradient**: Mise à jour itérative
   $$\boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \eta \cdot \nabla_{\boldsymbol{\theta}} \mathcal{L}$$

5. **Frontière de décision**: Un hyperplan $\boldsymbol{\theta}^\top \mathbf{x} = 0$

Ces mêmes idées se généralisent aux réseaux de neurones, où la régression logistique devient la couche de sortie pour la classification.

---

**Pour aller plus loin**: [Chapitre 3: Classification linéaire](https://pierrelux.github.io/mlbook/ch3_classification)